In [1]:
import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Twist
import ipywidgets as widgets
from IPython.display import display, clear_output
import threading
import time

class VelocityPublisher(Node):
    def __init__(self):
        super().__init__('velocity_publisher')
        self.publisher_ = self.create_publisher(Twist, '/cmd_vel', 10)
        self.twist = Twist()
        self.output = widgets.Output()
        self.running = True
        
        # Display widgets for current values
        self.linear_x_display = widgets.FloatText(value=0.0, description='Linear X:')
        self.linear_y_display = widgets.FloatText(value=0.0, description='Linear Y:')
        self.angular_z_display = widgets.FloatText(value=0.0, description='Angular Z:')
        
        # Frequency control
        self.freq_label = widgets.Label(value='Topic freq:')
        self.freq_box = widgets.FloatText(value=10.0, description='Hz')
        
        # Step size control
        self.step_label = widgets.Label(value='Step sizes:')
        self.linear_step_box = widgets.FloatText(value=0.05, description='Linear Step:')
        self.angular_step_box = widgets.FloatText(value=0.01, description='Angular Step:')
        
        # Zero velocity button
        self.zero_button = widgets.Button(description='Zero Velocity')
        self.zero_button.on_click(self.zero_velocity)
        
        display(self.linear_x_display, self.linear_y_display, self.angular_z_display)
        display(widgets.HBox([self.freq_label, self.freq_box]))
        display(widgets.HBox([self.step_label, self.linear_step_box, self.angular_step_box]))
        display(self.zero_button)
        
        # Start publishing in a separate thread
        self.publish_thread = threading.Thread(target=self.publish_velocity, daemon=True)
        self.publish_thread.start()
        
    def update_velocity(self, linear_x=0.0, linear_y=0.0, angular_z=0.0):
        self.twist.linear.x += linear_x
        self.twist.linear.y += linear_y
        self.twist.angular.z += angular_z
        
        # Update displayed values
        self.linear_x_display.value = self.twist.linear.x
        self.linear_y_display.value = self.twist.linear.y
        self.angular_z_display.value = self.twist.angular.z
        
        with self.output:
            clear_output(wait=True)
            print(f'Current Velocity:\nLinear: x={self.twist.linear.x}, y={self.twist.linear.y}\nAngular: z={self.twist.angular.z}')
        
    def zero_velocity(self, _):
        self.twist = Twist()
        self.update_velocity()
        self.get_logger().info('Velocity reset to zero.')
        
    def publish_velocity(self):
        while self.running:
            self.publisher_.publish(self.twist)
            time.sleep(1.0 / max(1.0, self.freq_box.value))
        
# Start ROS2 node
rclpy.init()
node = VelocityPublisher()

# Run ROS2 spin in a separate thread
def ros_spin():
    rclpy.spin(node)

spin_thread = threading.Thread(target=ros_spin, daemon=True)
spin_thread.start()

# Define buttons
button_x_plus = widgets.Button(description='X +')
button_x_minus = widgets.Button(description='X -')
button_y_plus = widgets.Button(description='Y +')
button_y_minus = widgets.Button(description='Y -')
button_angular_plus = widgets.Button(description='Yaw +')
button_angular_minus = widgets.Button(description='Yaw -')

# Define button actions
button_x_plus.on_click(lambda b: node.update_velocity(linear_x=node.linear_step_box.value))
button_x_minus.on_click(lambda b: node.update_velocity(linear_x=-node.linear_step_box.value))
button_y_plus.on_click(lambda b: node.update_velocity(linear_y=node.linear_step_box.value))
button_y_minus.on_click(lambda b: node.update_velocity(linear_y=-node.linear_step_box.value))
button_angular_plus.on_click(lambda b: node.update_velocity(angular_z=node.angular_step_box.value))
button_angular_minus.on_click(lambda b: node.update_velocity(angular_z=-node.angular_step_box.value))

# Display buttons
display(widgets.HBox([button_x_minus, button_x_plus]))
display(widgets.HBox([button_y_minus, button_y_plus]))
display(widgets.HBox([button_angular_minus, button_angular_plus]))
display(node.output)


FloatText(value=0.0, description='Linear X:')

FloatText(value=0.0, description='Linear Y:')

FloatText(value=0.0, description='Angular Z:')

Button(description='Zero Velocity', style=ButtonStyle())

Output()

[INFO] [1743073377.118681351] [velocity_publisher]: Velocity reset to zero.
[INFO] [1743073390.708703691] [velocity_publisher]: Velocity reset to zero.
